# 01-national-news-cleaning

See [project guide](../../README.md) and [data requirements](../../data/README.md) before execution. Workspace: `data/national-news/`. External inputs are not included. Run cells in order; model fitting and network collection are not run during repository checks.


In [ ]:
from pathlib import Path
import sys
import os
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'project_paths.py').is_file())
sys.path.insert(0, str(PROJECT_ROOT))
from project_paths import workspace
os.chdir(workspace('national-news'))


In [ ]:
%pprint
import os
import re
import time
import datetime
import warnings
import numpy as np 
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns; sns.set()

from tqdm import tqdm

%config InlineBackend.figure_format = 'retina'
# Keep warnings visible when checking the research environment.

In [ ]:
# Locate folder
os.chdir(workspace('national-news'))
path = workspace('national-news')      

# Overview

In [ ]:
%%time
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"
tw_pitt =  pd.read_csv('covid-news-en-us-clean-keyonly.csv')
tw_pitt['Date'] = pd.to_datetime(tw_pitt.Date, format="%Y-%m-%d").dt.date

In [ ]:
def update_keywords(tags):
    result = ' '.join(list(set(tags.replace('[','').replace(']','').replace('\'','').lower().split(', '))))      
    return result  
def update_hashtag(tags):
    try:
        result = ', '.join(set([' '.join( re.findall('[A-Z][^A-Z]*', word)) for word in 
             tags.replace('[','').replace(']','').replace('\'','').replace('#','').split(', ')])).lower()
       
        return result
    except:
        return ''

In [ ]:
'''
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"
tw_pitt =  pd.read_csv('covid-news-en-us.csv')
tw_pitt['time'] = pd.to_datetime(tw_pitt.time, format="%Y-%m-%d").dt.date
tw_pitt = tw_pitt.rename(columns={'time':'Date'})
tw_pitt = tw_pitt.sort_values(['Date']).reset_index(drop=True)
tw_pitt = tw_pitt.drop(columns=['language', 'country', 'social_shares_count', 'sentiment', 'word_count', 'entities', 'hashtag'])

tw_pitt.keyword = tw_pitt.keyword.apply(update_keywords)
tw_pitt.to_csv('covid-news-en-us-clean.csv', index=False)
tw_pitt[['Date','keyword','scopes']].to_csv('covid-news-en-us-clean-keyonly.csv', index=False)
'''

In [ ]:
# check empty cells
for col in tw_pitt.columns:
    display(tw_pitt[tw_pitt[col].isnull()])

# Cleaning - Text

In [ ]:
import nltk
from tqdm import tqdm
from nltk.corpus import stopwords as stpw
from nltk.stem.wordnet import WordNetLemmatizer
from nltk import word_tokenize, pos_tag # nltk.download('averaged_perceptron_tagger')
 
# lemmatization
lemma = WordNetLemmatizer() 

# stop words
def add_stopwords(wordlst):
    path = workspace('national-news') + 'extra_stopwords_en.txt'   
    with open(path, 'a') as f:
        for word in wordlst:
            f.write('\n'+word)
    f.close()
    return
def update_stopwords():
    from nltk.corpus import stopwords as stpw
    stopwords = []
    path = workspace('national-news')
    with open(path+"stopwords_en.txt", "r") as f1:
        for stopword in f1.readlines():
            stopwords.append(stopword.strip('\n'))
        
    with open(path+"extra_stopwords_en.txt", "r") as f2:
        for stopword in f2.readlines():
            stopwords.append(stopword.strip('\n'))

    stopwords.extend(stpw.words('english'))
    stopwords = set(stopwords)
    return stopwords
stopwords = update_stopwords()

In [ ]:
def filter_words(temp):
    temp = temp.lower()  
    # fist filter
    temp = [w for w in temp.split(' ') if not w in stopwords]
    temp = " ".join(word for word in temp)
    # to avoid removing contractions in english
    temp = re.sub("'", '', temp) 
    # Removing hashtags and mentions
    temp = re.sub("@[A-Za-z0-9_]+",'', temp)
    temp = re.sub("#[A-Za-z0-9_]+",'', temp)
    # Removing links
    temp = re.sub(r"http\S+", '', temp)
    temp = re.sub(r"www.\S+", '', temp)
    # Removing punctuations
    temp = re.sub('[.()!?]', ' ', temp)
    temp = re.sub(r'\[.*?\]',' ', temp)
    return temp
def filter_words2(temp):
    # Furter clean
    # temp = temp.lower()  
    temp = [w for w in temp.split(' ') if not w in stopwords]
    temp = [w for w in temp if len(w)>2]
    temp = " ".join(word for word in temp)
    return temp
def replace_words(temp):
    # temp = temp.lower()
    temp = temp.replace('covid 19' , 'covid-19')
    temp = temp.replace('covid19' , 'covid-19')
    temp = temp.replace('hong kong' , 'hongkong')
    temp = temp.replace('wu han' , 'wuhan') 
    return temp

def convert_postag(word2pos):
    # word2pos = ('eating', 'VBG')
    word = word2pos[0]
    tag = word2pos[1]
    
    if tag.startswith('J'):
        tag = 'a'
    elif tag.startswith('V'):
        tag = 'v'
    elif tag.startswith('N'):
        tag = 'n'
    elif tag.startswith('R'):
        tag = 'r'
    else:
        tag = 'skip'    
    return (word,tag)
def remove_tense(sentence):
    word2pos_lst = [convert_postag(word2pos) for word2pos in pos_tag(word_tokenize(sentence))]
    word2pos_lst = [word2pos for word2pos in word2pos_lst if not word2pos[1] == 'skip']
    sentence = ' '.join([WordNetLemmatizer().lemmatize(word2pos[0],word2pos[1]) for word2pos in word2pos_lst ])
    return sentence

In [ ]:
def filter_noneNVtag(word2pos):
    # word2pos = ('eating', 'VBG')
    word = word2pos[0]
    tag = word2pos[1]
    
    if tag.startswith('V') or tag.startswith('N'):
        return (word,tag)
    else:
        tag = 'skip'    
    return (word,tag)
def none_and_verb(temp):
    wordlst = word_tokenize(temp)
    word2pos_lst = pos_tag(wordlst)
    word2pos_lst = [filter_noneNVtag(word2pos) for word2pos in word2pos_lst]
    word2pos_lst = [word2pos for word2pos in word2pos_lst if not word2pos[1] == 'skip']
    wordlst = [word2pos[0] for word2pos in word2pos_lst ]
    temp = " ".join(wordlst)
    return temp

def filter_noneNtag(word2pos):
    # word2pos = ('eating', 'VBG')
    word = word2pos[0]
    tag = word2pos[1]
    
    if tag.startswith('N'):
        return (word,tag)
    else:
        tag = 'skip'    
    return (word,tag)
def none(temp):
    wordlst = word_tokenize(temp)
    word2pos_lst = pos_tag(wordlst)
    word2pos_lst = [filter_noneNtag(word2pos) for word2pos in word2pos_lst]
    word2pos_lst = [word2pos for word2pos in word2pos_lst if not word2pos[1] == 'skip']
    wordlst = [word2pos[0] for word2pos in word2pos_lst ]
    temp = " ".join(wordlst)
    return temp

In [ ]:
tw_pitt['cleaned_key'] = tw_pitt.keyword.apply(lambda temp: replace_words(filter_words(temp)))
tw_pitt['cleaned_key'] = tw_pitt.cleaned_key.apply(filter_words2)
tw_pitt['cleaned_key'] = tw_pitt.cleaned_key.apply(lambda x: ' '.join(dict.fromkeys(x.split())) )

In [ ]:
tw_pitt

In [ ]:
# tw_pitt.to_csv('covid-news-en-us-clean-keyonly.csv', index=False)

In [ ]:
%%time
r_t = tw_pitt.cleaned_key.apply(remove_tense)

In [ ]:
%%time
cleaned_verb_noun = r_t.apply(lambda x: none_and_verb(x))

In [ ]:
%%time
cleaned_noun = r_t.apply(lambda x: none(x))

In [ ]:
tw_pitt['cleaned_key'] = r_t
tw_pitt['cleaned_key_VN'] = cleaned_verb_noun
tw_pitt['cleaned_key_N'] = cleaned_noun

In [ ]:
tw_pitt

In [ ]:
tw_pitt.scopes.value_counts()[:60]

In [ ]:
# tw_pitt.to_csv('covid-news-en-us-clean-keyonly.csv', index=False)

# Filter by Word Count 

In [ ]:
# Continue with the in-memory cleaned table.
# tw_pitt = tw_pitt.fillna('')
tw_pitt 

In [ ]:
# Define word count function
def count_words(wordslst):
    df = pd.DataFrame( {'Word': wordslst, 'Count':np.zeros(len(wordslst))} )
    wordcounts = df.groupby('Word').agg({'Count':np.size}).sort_values(by = 'Count', ascending = False)
    wordcounts['Prop%'] = (wordcounts.Count/len(wordslst))*100
    wordcounts = wordcounts.reset_index()
    
    # sort_values(by=‘ColumnName’, axis=0 , ascending = True , inplace = False, na_position = ‘last’ )
    # the column name of agg must be pre-determined
    # df = df.reset_index() can maintain the original index as a column and add a new index starting from 0

    return wordcounts
def checkWords(word, wordcounts):
    # wordcounts is the df defined from above using count_words( wordslst )     
    return wordcounts[ wordcounts['Word'] == word ]
def display_wordcounts(Type, rank, wordcounts, display=True):
    n = wordcounts.shape[0]
    
    if rank==-1:
        temp = wordcounts
        rank = wordcounts.shape[0]
    elif Type == 'top':
        temp = wordcounts.iloc[:rank,:]
    elif Type == 'bottom':
        temp = wordcounts.iloc[n-rank:,:]
        
    lst = [(temp.iloc[i,0],temp.iloc[i,1],temp.iloc[i,2]) for i in range(rank)]
    
    if display:     
        for i in range(rank):
            print("%-20s\t%-10d\t%.7f"%(lst[i][0],lst[i][1],lst[i][2]))     
    return lst

In [ ]:
all_words = ' '.join(list(tw_pitt.cleaned_key)).split()
wordcounts = count_words(all_words)
display(wordcounts)

In [ ]:
wordcounts[wordcounts['Prop%']>=top_prop]

In [ ]:
top_prop = 0.01
print(wordcounts[wordcounts['Prop%']>=top_prop]['Prop%'].sum())
print(wordcounts[wordcounts['Prop%']>=top_prop]['Count'].sum())
print()
top = display_wordcounts('top', -1, wordcounts[wordcounts['Prop%']>=top_prop])

In [ ]:
bottom_prop = 0.01
print(wordcounts[wordcounts['Prop%']<bottom_prop]['Prop%'].sum())
print(wordcounts[wordcounts['Prop%']<bottom_prop]['Count'].sum())
print()
bottom = display_wordcounts('top', -1, wordcounts[wordcounts['Prop%']<bottom_prop])

In [ ]:
bottom_words= [x[0] for x in bottom] 
add_stopwords(bottom_words)
stopwords = update_stopwords()

In [ ]:
# tw_pitt.to_csv('covid-news-en-us-clean-keyonly.csv', index=False)

# Text/Key By Day

In [ ]:
%%time
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"
# Continue with the in-memory cleaned table.
tw_pitt['Date'] = pd.to_datetime(tw_pitt.Date, format="%Y-%m-%d").dt.date
tw_pitt = tw_pitt.fillna('')

In [ ]:
# Data Type 
tw_pitt_text2day  = tw_pitt[['Date','cleaned_key']]        # All text
tw_pitt_VN2day    = tw_pitt[['Date','cleaned_key_VN']]     # All verb and none
tw_pitt_N2day     = tw_pitt[['Date','cleaned_key_N']]      # All verb and none

## Combine text in one day

In [ ]:
def sumstr(df):
    return ' '.join(list(df.iloc[:,1]))

tw_pitt_text2day  = tw_pitt_text2day.groupby('Date')[tw_pitt_text2day.columns[1]].agg(' '.join)
tw_pitt_VN2day    = tw_pitt_VN2day.groupby('Date')[tw_pitt_VN2day.columns[1]].agg(' '.join)
tw_pitt_N2day     = tw_pitt_N2day.groupby('Date')[tw_pitt_N2day.columns[1]].agg(' '.join)

tw_pitt_text2day  = pd.DataFrame(tw_pitt_text2day,columns=['Content'])
tw_pitt_VN2day    = pd.DataFrame(tw_pitt_VN2day,columns=['Content'])
tw_pitt_N2day     = pd.DataFrame(tw_pitt_N2day ,columns=['Content'])

In [ ]:
tw_pitt_text2day.to_csv('Doc1_key2day.csv', index=True)
tw_pitt_VN2day.to_csv('Doc2_key_VN2day.csv', index=True)
tw_pitt_N2day.to_csv('Doc3_key_N2day.csv', index=True)

## Check empty data

In [ ]:
def check_empty_data(df, display_=False):
    start_day = df.index[0]
    end_day = df.index[-1]
    structure = pd.DataFrame(index = pd.date_range(start_day, end_day), columns=['Content'])
    structure['Content'] = df['Content']
    
    if display_:
        display(structure[structure.iloc[:,0].isna()])
    return structure 

In [ ]:
tw_pitt_text2day_ = check_empty_data(tw_pitt_text2day, display_=True)
tw_pitt_VN2day_ = check_empty_data(tw_pitt_VN2day, display_=True)
tw_pitt_N2day_ = check_empty_data(tw_pitt_N2day, display_=True)

tw_pitt_text2day_.iloc[53:,:].to_csv('Doc1_key2day.csv', index=True)
tw_pitt_VN2day_.iloc[53:,:].to_csv('Doc2_key_VN2day.csv', index=True)
tw_pitt_N2day_.iloc[53:,:].to_csv('Doc3_key_N2day.csv', index=True)